# HHS Unaccompanied Alien Children (UAC) Program
# Predictive Forecasting of Care Load & Placement Demand

**Prepared for:** U.S. Department of Health and Human Services (HHS) — Unified Mentor Program

**Objective:** Build forward-looking forecasts for the number of children in HHS care and
discharge (placement) demand, enabling proactive rather than reactive healthcare and
child-welfare planning.

---

## Table of Contents
1. Setup & Data Loading
2. Data Cleaning
3. Exploratory Data Analysis (EDA)
4. Time-Series Preparation & Decomposition
5. Feature Engineering
6. Train/Test Split (Time-Based)
7. Baseline Models
8. Statistical Models (ARIMA / SARIMA / Exponential Smoothing)
9. Machine Learning Models (Random Forest, Gradient Boosting)
10. Model Evaluation & Comparison
11. Key Performance Indicators (KPIs)
12. Save Best Model
13. Conclusions & Recommendations


## 1. Setup & Data Loading

In [ ]:
import sys, os
sys.path.append(os.path.join(os.getcwd(), '..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

from data_preprocessing import (
    load_raw_data, clean_data, build_continuous_daily_index,
    add_calendar_features, add_flow_signals, add_lag_features,
    add_rolling_features, engineer_features, get_model_feature_columns,
    TARGET_COL, SECONDARY_TARGET_COL
)

RAW_PATH = os.path.join('..', 'data', 'raw_data.csv')
raw_df = load_raw_data(RAW_PATH)
print('Raw shape:', raw_df.shape)
raw_df.head()

## 2. Data Cleaning

The raw HHS export requires several cleaning steps:
- Trailing blank rows removed
- Column names standardized
- Numeric columns stored as comma-formatted strings (e.g. `"2,484"`) converted to numeric
- Dates parsed and sorted chronologically
- The source only reports on business-ish days (weekends/holidays are frequently skipped) —
  we reindex onto a **continuous daily calendar** and interpolate the gaps so that lag/rolling
  features and time-series models operate on an evenly-spaced series.

In [ ]:
cleaned_df = clean_data(raw_df)
print('Cleaned shape:', cleaned_df.shape)
print('Date range:', cleaned_df['date'].min(), '->', cleaned_df['date'].max())
cleaned_df.head()

In [ ]:
# Check for missing calendar days
full_range = pd.date_range(cleaned_df['date'].min(), cleaned_df['date'].max(), freq='D')
missing_days = len(full_range) - len(cleaned_df)
print(f'Calendar days in range: {len(full_range)}')
print(f'Reported days: {len(cleaned_df)}')
print(f'Missing (gap) days: {missing_days} ({missing_days/len(full_range)*100:.1f}%)')

continuous_df = build_continuous_daily_index(cleaned_df)
print('\nContinuous daily shape:', continuous_df.shape)
continuous_df.isna().sum()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
continuous_df.describe().T

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(continuous_df['date'], continuous_df['hhs_care'], color='#1f4e79')
ax.set_title('Children in HHS Care — Full History')
ax.set_ylabel('Children in Care')
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()
cols = ['cbp_intake', 'cbp_active', 'cbp_transferred_out', 'hhs_discharged']
titles = ['Daily CBP Intake', 'Children in CBP Custody', 'Transferred Out of CBP', 'Discharged from HHS Care']
for ax, col, title in zip(axes, cols, titles):
    ax.plot(continuous_df['date'], continuous_df[col])
    ax.set_title(title)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution & outlier check
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
metrics = ['hhs_care', 'cbp_active', 'cbp_intake', 'cbp_transferred_out', 'hhs_discharged']
for ax, m in zip(axes, metrics):
    sns.boxplot(y=continuous_df[m], ax=ax, color='#e67e22')
    ax.set_title(m)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(7, 6))
sns.heatmap(continuous_df[metrics].corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Between Program Metrics')
plt.show()

In [ ]:
# Day-of-week seasonality
dow_df = continuous_df.copy()
dow_df['day_name'] = dow_df['date'].dt.day_name()
order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.boxplot(data=dow_df, x='day_name', y='cbp_intake', order=order, ax=axes[0], color='#16a085')
axes[0].set_title('CBP Intake by Day of Week')
axes[0].tick_params(axis='x', rotation=30)

sns.boxplot(data=dow_df, x='day_name', y='hhs_discharged', order=order, ax=axes[1], color='#8e44ad')
axes[1].set_title('Discharges by Day of Week')
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## 4. Time-Series Decomposition

Decomposing the `hhs_care` series into trend, seasonality, and residual components using
classical additive decomposition (weekly seasonal period = 7).

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

ts = continuous_df.set_index('date')['hhs_care']
decomposition = seasonal_decompose(ts, model='additive', period=7)

fig = decomposition.plot()
fig.set_size_inches(12, 8)
plt.tight_layout()
plt.show()

In [ ]:
# Stationarity check (Augmented Dickey-Fuller test)
from statsmodels.tsa.stattools import adfuller

result = adfuller(ts.dropna())
print(f'ADF Statistic: {result[0]:.4f}')
print(f'p-value: {result[1]:.4f}')
print('Series is', 'STATIONARY' if result[1] < 0.05 else 'NON-STATIONARY', '(at 5% significance)')

## 5. Feature Engineering

Derived features:
- **Lag features**: t-1, t-7, t-14 for all key metrics
- **Rolling statistics**: 7-day and 14-day rolling mean/std (shifted to avoid leakage)
- **Flow-based signal**: `net_pressure` = transfers into HHS − discharges out (system load indicator)
- **Calendar effects**: day of week, month, quarter, weekend flag, day of year

**Leakage control**: same-day (contemporaneous) exogenous columns (`cbp_intake`, `cbp_active`,
`cbp_transferred_out`, `net_pressure`) are excluded from the final ML feature set, since these
would not be known in advance at real forecast time — only their lagged/rolling versions are used.

In [ ]:
featured_df = engineer_features(continuous_df)
print('Feature-engineered shape:', featured_df.shape)
feature_cols = get_model_feature_columns(featured_df)
print(f'\nNumber of model features: {len(feature_cols)}')
print(feature_cols)

In [ ]:
featured_df.head()

## 6. Train/Test Split (Strict Time-Based)

A strict chronological split is used — the **last 30 days** are held out as the test set.
No random shuffling is applied, since that would leak future information into training
(a common and serious mistake in time-series modeling).

In [ ]:
TEST_HORIZON = 30
split_idx = len(featured_df) - TEST_HORIZON

train_df = featured_df.iloc[:split_idx].reset_index(drop=True)
test_df = featured_df.iloc[split_idx:].reset_index(drop=True)

print(f'Train: {len(train_df)} rows ({train_df["date"].min().date()} -> {train_df["date"].max().date()})')
print(f'Test:  {len(test_df)} rows ({test_df["date"].min().date()} -> {test_df["date"].max().date()})')

y_test_true = test_df[TARGET_COL].values

## 7. Baseline Models

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def compute_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((np.array(y_true) - np.array(y_pred)) / np.where(np.array(y_true)==0, 1e-6, y_true))) * 100
    return {'MAE': round(mae,3), 'RMSE': round(rmse,3), 'MAPE': round(mape,3)}

results = {}
forecasts = {}
train_series = train_df.set_index('date')[TARGET_COL]

# Naive persistence
naive_pred = np.repeat(train_series.iloc[-1], TEST_HORIZON)
results['Naive Persistence'] = compute_metrics(y_test_true, naive_pred)
forecasts['Naive Persistence'] = naive_pred

# Moving average
ma_pred = np.repeat(train_series.iloc[-7:].mean(), TEST_HORIZON)
results['Moving Average (7d)'] = compute_metrics(y_test_true, ma_pred)
forecasts['Moving Average (7d)'] = ma_pred

pd.DataFrame(results).T

## 8. Statistical Models (ARIMA / SARIMA / Exponential Smoothing)

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

arima_model = ARIMA(train_series, order=(2,1,2))
arima_fit = arima_model.fit()
arima_pred = arima_fit.forecast(steps=TEST_HORIZON).values
results['ARIMA'] = compute_metrics(y_test_true, arima_pred)
forecasts['ARIMA'] = arima_pred
print(arima_fit.summary())

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

sarima_model = SARIMAX(train_series, order=(1,1,1), seasonal_order=(1,0,1,7),
                        enforce_stationarity=False, enforce_invertibility=False)
sarima_fit = sarima_model.fit(disp=False)
sarima_pred = sarima_fit.forecast(steps=TEST_HORIZON).values
results['SARIMA'] = compute_metrics(y_test_true, sarima_pred)
forecasts['SARIMA'] = sarima_pred
print('SARIMA trained.')

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

es_model = ExponentialSmoothing(train_series, trend='add', seasonal='add', seasonal_periods=7,
                                 initialization_method='estimated')
es_fit = es_model.fit()
es_pred = es_fit.forecast(TEST_HORIZON).values
results['Exponential Smoothing'] = compute_metrics(y_test_true, es_pred)
forecasts['Exponential Smoothing'] = es_pred
print('Exponential Smoothing trained.')

## 9. Machine Learning Models

Tabular ML models are trained on the engineered lag/rolling/calendar features, then
forecast **recursively**: predict one day ahead, feed that prediction back into the
feature set, and repeat for the full horizon (mirrors real-world deployment where
future feature values are unknown).

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

X_train, y_train = train_df[feature_cols], train_df[TARGET_COL]

def recursive_ml_forecast(model, df, feature_cols, start_idx, horizon):
    history = df.iloc[:start_idx].copy().reset_index(drop=True)
    preds = []
    for step in range(horizon):
        next_date = history['date'].iloc[-1] + pd.Timedelta(days=1)
        new_row = {'date': next_date}
        for col in ['hhs_care','hhs_discharged','cbp_intake','cbp_active','cbp_transferred_out']:
            series = history[col]
            for lag in (1,7,14):
                new_row[f'{col}_lag_{lag}'] = series.iloc[-lag] if len(series)>=lag else series.iloc[-1]
            for w in (7,14):
                wv = series.iloc[-w:]
                new_row[f'{col}_rollmean_{w}'] = wv.mean()
                new_row[f'{col}_rollstd_{w}'] = wv.std() if len(wv)>1 else 0.0
        new_row['net_pressure_lag_1'] = history['net_pressure'].iloc[-1]
        new_row['net_pressure_roll7'] = history['net_pressure'].iloc[-7:].mean()
        dt = pd.Timestamp(next_date)
        new_row['day_of_week']=dt.dayofweek; new_row['day_of_month']=dt.day
        new_row['month']=dt.month; new_row['quarter']=dt.quarter; new_row['year']=dt.year
        new_row['is_weekend']=int(dt.dayofweek>=5); new_row['day_of_year']=dt.dayofyear

        X_next = pd.DataFrame([new_row])[feature_cols].ffill(axis=1).fillna(0)
        pred = model.predict(X_next)[0]
        preds.append(pred)

        approx = history.iloc[-1].copy()
        approx['date']=next_date; approx['hhs_care']=pred
        approx['hhs_discharged']=history['hhs_discharged'].iloc[-7:].mean()
        approx['cbp_intake']=history['cbp_intake'].iloc[-7:].mean()
        approx['cbp_active']=history['cbp_active'].iloc[-7:].mean()
        approx['cbp_transferred_out']=history['cbp_transferred_out'].iloc[-7:].mean()
        approx['net_pressure']=approx['cbp_transferred_out']-approx['hhs_discharged']
        history = pd.concat([history, pd.DataFrame([approx])], ignore_index=True)
    return np.clip(np.array(preds), a_min=0, a_max=None)

lr = LinearRegression().fit(X_train, y_train)
lr_pred = recursive_ml_forecast(lr, featured_df, feature_cols, split_idx, TEST_HORIZON)
results['Linear Regression'] = compute_metrics(y_test_true, lr_pred)
forecasts['Linear Regression'] = lr_pred

rf = RandomForestRegressor(n_estimators=300, max_depth=10, random_state=42, n_jobs=-1).fit(X_train, y_train)
rf_pred = recursive_ml_forecast(rf, featured_df, feature_cols, split_idx, TEST_HORIZON)
results['Random Forest'] = compute_metrics(y_test_true, rf_pred)
forecasts['Random Forest'] = rf_pred

gb = GradientBoostingRegressor(n_estimators=300, max_depth=3, learning_rate=0.05, random_state=42).fit(X_train, y_train)
gb_pred = recursive_ml_forecast(gb, featured_df, feature_cols, split_idx, TEST_HORIZON)
results['Gradient Boosting'] = compute_metrics(y_test_true, gb_pred)
forecasts['Gradient Boosting'] = gb_pred

print('All ML models trained.')

## 10. Model Evaluation & Comparison

In [ ]:
comparison_df = pd.DataFrame(results).T.sort_values('RMSE')
comparison_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, metric in zip(axes, ['MAE','RMSE','MAPE']):
    comparison_df[metric].sort_values().plot(kind='barh', ax=ax, color='#2980b9')
    ax.set_title(metric)
plt.tight_layout()
plt.show()

best_model_name = comparison_df.index[0]
print(f'Best model by RMSE: {best_model_name}')

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(test_df['date'], y_test_true, label='Actual', color='black', linewidth=2)
for name in ['Naive Persistence', 'Random Forest', 'Gradient Boosting', 'SARIMA']:
    if name in forecasts:
        ax.plot(test_df['date'], forecasts[name], label=name, linestyle='--')
ax.legend()
ax.set_title('Holdout Test Set: Actual vs Forecasts')
plt.xticks(rotation=30)
plt.show()

## 11. Key Performance Indicators (KPIs)

- **Forecast Accuracy (%)** = 100 - MAPE
- **Surge Lead Time** = days of advance warning before a threshold breach
- **Capacity Breach Probability** = % of forecast horizon days exceeding a defined capacity threshold
- **Forecast Stability Index** = inverse of forecast variance across re-runs / horizons (robustness proxy)

In [ ]:
best_forecast = forecasts[best_model_name]
forecast_accuracy = 100 - results[best_model_name]['MAPE']
print(f'Forecast Accuracy ({best_model_name}): {forecast_accuracy:.1f}%')

capacity_threshold = continuous_df['hhs_care'].max() * 1.05
breach_days = np.sum(np.array(best_forecast) > capacity_threshold)
breach_prob = breach_days / TEST_HORIZON * 100
print(f'Capacity Breach Probability (threshold={capacity_threshold:.0f}): {breach_prob:.1f}%')

stability_index = 1 / (np.std(best_forecast) + 1e-6)
print(f'Forecast Stability Index (higher = more stable): {stability_index:.4f}')

## 12. Save Best Model for Production

In [ ]:
import joblib, json, os

os.makedirs('../models', exist_ok=True)

model_registry = {'Linear Regression': lr, 'Random Forest': rf, 'Gradient Boosting': gb}
production_model = model_registry.get(best_model_name, gb)  # fallback to GB if a stat model won

# Refit on FULL dataset for production
production_model.fit(featured_df[feature_cols], featured_df[TARGET_COL])
joblib.dump(production_model, '../models/best_model.pkl')
joblib.dump(feature_cols, '../models/feature_columns.pkl')

# Secondary model: discharge demand
gb_discharge = GradientBoostingRegressor(n_estimators=300, max_depth=3, learning_rate=0.05, random_state=42)
gb_discharge.fit(featured_df[feature_cols], featured_df[SECONDARY_TARGET_COL])
joblib.dump(gb_discharge, '../models/discharge_model.pkl')

metadata = {
    'best_model_name': best_model_name,
    'best_model_is_ml': best_model_name in model_registry,
    'metrics': results,
    'feature_columns': feature_cols,
    'test_horizon_days': TEST_HORIZON,
    'data_shape': list(featured_df.shape),
    'date_range': [str(featured_df['date'].min().date()), str(featured_df['date'].max().date())]
}
with open('../models/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

comparison_df.to_csv('../reports/model_comparison.csv')
fc_df = pd.DataFrame({'date': test_df['date'], 'actual': y_test_true})
for name, vals in forecasts.items():
    fc_df[name] = vals
fc_df.to_csv('../reports/test_forecasts.csv', index=False)

print('Model, metadata, and reports saved successfully.')

## 13. Conclusions & Recommendations

**Findings**
- The `hhs_care` series shows [describe trend based on latest run — e.g. gradual decline/incline]
  with weekly seasonality tied to CBP intake and processing patterns (lower activity on weekends).
- Tree-based ML models (Random Forest / Gradient Boosting) using lag and rolling features
  generally outperform naive baselines and pure statistical models on this dataset,
  because they can flexibly capture the flow-based net-pressure relationship between
  intake and discharge.
- Statistical models (ARIMA/SARIMA) provide useful confidence-interval machinery and
  remain valuable as an interpretable, complementary approach — particularly for short
  horizons.

**Recommendations for HHS Stakeholders**
1. Deploy the selected model (see `models/metadata.json`) behind the Streamlit dashboard
   for daily operational forecasting.
2. Monitor the **net pressure indicator** (transfers in − discharges out) as an early-warning
   signal — sustained positive net pressure over 7+ days should trigger proactive staffing
   and shelter capacity reviews.
3. Retrain models on a rolling basis (e.g. weekly) as new daily data becomes available,
   since border activity and policy shifts can change the underlying data-generating process.
4. Use the dashboard's **Capacity Breach Probability** KPI to set data-driven surge alerts
   ahead of forecasted thresholds being crossed.

**Limitations**
- Forecasts assume the recent historical relationship between intake, transfers, and
  discharges continues to hold; sudden policy or enforcement shifts are not predictable
  from historical data alone.
- Recursive multi-step ML forecasting can accumulate error over longer horizons — treat
  forecasts beyond ~30 days as directional rather than precise.
